# Week 12 Portfolio Milestone 1: Build and Validate a Healthcare Database

## Google Colab notebook

This notebook combines the Week 12 Python loading workflow, SQLite schema work, validation checks, and reproducibility test. Use only the supplied synthetic seed data.

Save your own working copy in Google Drive before editing: **File → Save a copy in Drive**. Colab runtime storage is temporary, so download both required submission files before ending the session.

## Assignment steps

1. Refine your Week 9 ERD and data dictionary.
2. Complete the database schema using primary keys, foreign keys, `NOT NULL`, and appropriate `CHECK` or `UNIQUE` constraints.
3. Create a new SQLite database named `community_followup.sqlite`.
4. Enable foreign keys and run the schema.
5. Load all five supplied seed CSV files in parent-to-child order.
6. Run the required validation checks.
7. Recreate the database from a clean file to confirm that the schema and loading process are reproducible.

Complete the notebook from top to bottom. Cells marked **TODO** require your work. Do not enter real patient or workplace data.


## Colab setup: upload the five seed CSV files

Download the Week 12 `seed_data` files from the course materials. When the next cell opens the Colab file picker, select all five files at the same time:

- `clinics.csv`
- `patients.csv`
- `encounters.csv`
- `referrals.csv`
- `followups.csv`

The setup code places them in a `seed_data` folder. If the notebook is running locally beside the supplied folder, the upload prompt is skipped.


In [ ]:
from pathlib import Path
import csv
import sqlite3
import zipfile

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

WORK_DIR = Path("/content") if IN_COLAB else Path.cwd()
SEED_DIR = WORK_DIR / "seed_data"
SEED_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_SEED_FILES = [
    "clinics.csv",
    "patients.csv",
    "encounters.csv",
    "referrals.csv",
    "followups.csv",
]

missing_files = [
    filename for filename in REQUIRED_SEED_FILES
    if not (SEED_DIR / filename).exists()
]

if missing_files and IN_COLAB:
    print("Select all five Week 12 seed CSV files.")
    uploaded = files.upload()
    for uploaded_name, uploaded_bytes in uploaded.items():
        destination = SEED_DIR / Path(uploaded_name).name
        destination.write_bytes(uploaded_bytes)

missing_files = [
    filename for filename in REQUIRED_SEED_FILES
    if not (SEED_DIR / filename).exists()
]

assert not missing_files, (
    "Missing seed files: " + ", ".join(missing_files)
)

print("Working folder:", WORK_DIR)
print("Seed folder:", SEED_DIR)
print("All five seed files are available.")


## Step 1: Refine your Week 9 ERD and data dictionary.

Compare your Week 9 ERD and data dictionary with the five supplied CSV headers. Your design must preserve the required identifiers and relationships:

- One clinic can have many encounters.
- One patient can have many encounters.
- One encounter can have many referrals.
- Each supplied referral has one follow-up record.

Update the ERD and data dictionary whenever the implemented schema changes. The data dictionary should document table name, field name, definition, data type, key or constraint, nullability, and an example value.


In [ ]:
# Inspect each source file before writing the schema.
seed_inventory = []

for filename in REQUIRED_SEED_FILES:
    path = SEED_DIR / filename
    with path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
    seed_inventory.append({
        "file": filename,
        "table": path.stem,
        "rows": len(rows),
        "columns": ", ".join(reader.fieldnames or []),
    })

seed_inventory_df = pd.DataFrame(seed_inventory)
display(seed_inventory_df)


### Step 1 documentation checkpoint

Before continuing, confirm:

- [ ] The ERD contains `clinics`, `patients`, `encounters`, `referrals`, and `followups`.
- [ ] The relationship direction and cardinality match the supplied identifiers.
- [ ] The data dictionary includes every seed-file field.
- [ ] The planned table and column names match the CSV headers so the loading code can use them.


## Step 2: Complete the database schema using primary keys, foreign keys, `NOT NULL`, and appropriate `CHECK` or `UNIQUE` constraints.

Complete the SQL string in the next cell. The `clinics` and `patients` tables reproduce the supplied schema starter. Add `CREATE TABLE` statements for `encounters`, `referrals`, and `followups` using their CSV headers.

Required relationships:

- `encounters.patient_id` references `patients.patient_id`.
- `encounters.clinic_id` references `clinics.clinic_id`.
- `referrals.encounter_id` references `encounters.encounter_id`.
- `followups.referral_id` references `referrals.referral_id` and is `UNIQUE` for the supplied one-to-one design.

Use `CHECK` constraints for the supplied stable categories or rules, such as priority, social-needs flag, outcome, and nonnegative contact attempts. Do not invent unsupported clinical rules.


In [ ]:
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE clinics (
    clinic_id TEXT PRIMARY KEY,
    clinic_name TEXT NOT NULL UNIQUE,
    region TEXT NOT NULL
);

CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY,
    birth_year INTEGER NOT NULL,
    preferred_language TEXT NOT NULL,
    social_needs_flag INTEGER NOT NULL CHECK (social_needs_flag IN (0, 1))
);

-- TODO: Create encounters using these CSV fields:
-- encounter_id, patient_id, clinic_id, encounter_date, visit_type
-- Add foreign keys to patients and clinics.

-- TODO: Create referrals using these CSV fields:
-- referral_id, encounter_id, service_type, referral_date, priority, due_date
-- Add a foreign key to encounters and a CHECK for priority.

-- TODO: Create followups using these CSV fields:
-- followup_id, referral_id, completion_date, outcome, contact_attempts
-- Make referral_id UNIQUE, add the foreign key to referrals, and add
-- appropriate CHECK constraints for outcome and contact_attempts.
"""

print(SCHEMA_SQL)


In [ ]:
SCHEMA_PATH = WORK_DIR / "schema.sql"
SCHEMA_PATH.write_text(SCHEMA_SQL.strip() + "\n", encoding="utf-8")
print("Schema saved to:", SCHEMA_PATH)


## Step 3: Create a new SQLite database named `community_followup.sqlite`.

The next cell deletes only the temporary database with this exact filename if it already exists, then creates a new empty database. Rerun this step whenever you need a clean rebuild.


In [ ]:
DB_PATH = WORK_DIR / "community_followup.sqlite"

try:
    connection.close()
except NameError:
    pass

if DB_PATH.exists():
    DB_PATH.unlink()

connection = sqlite3.connect(DB_PATH)
print("New database created:", DB_PATH)


## Step 4: Enable foreign keys and run the schema.

SQLite requires foreign-key enforcement to be enabled for each connection. Turn it on before creating or loading tables. Then execute the schema and verify that all five required tables exist.

If this step reports missing tables, return to Step 2 and complete the unfinished `CREATE TABLE` statements. Then rerun Steps 2–4.


In [ ]:
connection.execute("PRAGMA foreign_keys = ON")
foreign_keys_enabled = connection.execute("PRAGMA foreign_keys").fetchone()[0]
assert foreign_keys_enabled == 1, "Foreign-key enforcement is not enabled."

connection.executescript(SCHEMA_SQL)
connection.commit()

EXPECTED_TABLES = {
    "clinics", "patients", "encounters", "referrals", "followups"
}
table_names = {
    row[0]
    for row in connection.execute(
        """
        SELECT name
        FROM sqlite_schema
        WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name;
        """
    ).fetchall()
}

print("foreign_keys:", foreign_keys_enabled)
print("Tables created:", sorted(table_names))

missing_tables = EXPECTED_TABLES - table_names
assert not missing_tables, (
    "Complete the schema before continuing. Missing tables: "
    + ", ".join(sorted(missing_tables))
)


In [ ]:
# Inspect the SQL stored inside the database.
schema_inspection = pd.read_sql_query(
    """
    SELECT name, sql
    FROM sqlite_schema
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """,
    connection,
)
display(schema_inspection)


## Step 5: Load all five supplied seed CSV files in parent-to-child order.

The loading code adapts `Week12_Load_Seed_Data_Optional.py` for Colab. It reads each CSV with `csv.DictReader`, converts blank strings to `None`, and inserts rows with parameterized SQL.

Load order matters because foreign keys are active:

1. `clinics`
2. `patients`
3. `encounters`
4. `referrals`
5. `followups`

Run this cell once after creating a clean database. To reload, return to Step 3 and rebuild first.


In [ ]:
TABLE_LOAD_ORDER = [
    "clinics",
    "patients",
    "encounters",
    "referrals",
    "followups",
]

loaded_rows = {}

with connection:
    for table in TABLE_LOAD_ORDER:
        csv_path = SEED_DIR / f"{table}.csv"
        with csv_path.open(newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            rows = list(reader)

        columns = list(reader.fieldnames or [])
        placeholders = ", ".join("?" for _ in columns)
        column_sql = ", ".join(columns)
        insert_sql = (
            f"INSERT INTO {table} ({column_sql}) "
            f"VALUES ({placeholders})"
        )
        values = [
            [row[column] if row[column] != "" else None for column in columns]
            for row in rows
        ]
        connection.executemany(insert_sql, values)
        loaded_rows[table] = len(rows)

print("Rows read and inserted:")
display(pd.Series(loaded_rows, name="rows"))


In [ ]:
# Immediate loading checkpoint.
database_row_counts = {
    table: connection.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]
    for table in TABLE_LOAD_ORDER
}

load_check_df = pd.DataFrame({
    "source_rows": pd.Series(loaded_rows),
    "database_rows": pd.Series(database_row_counts),
})
load_check_df["matches"] = (
    load_check_df["source_rows"] == load_check_df["database_rows"]
)
display(load_check_df)

assert load_check_df["matches"].all(), "A table count does not match its CSV."
print("Every source row was loaded exactly once.")


## Step 6: Run the required validation checks.

The following cells implement the required checks from `Week12_Validation_Checklist.md`. The results remain visible in the completed notebook; no separate validation report is required.

Validation covers:

- Foreign-key enforcement and integrity.
- Required table names and expected row counts.
- Parent-child relationship joins.
- Allowed priority and social-needs values.
- Missing completion dates used to identify pending referrals.
- Reconciliation of the database with the seed CSVs.


In [ ]:
def query_df(sql, parameters=None, database_connection=None):
    """Run a read-only SQL query and return a pandas DataFrame."""
    active_connection = database_connection or connection
    return pd.read_sql_query(
        sql,
        active_connection,
        params=parameters,
    )


In [ ]:
# Structure and enforcement checks.
foreign_keys_result = query_df("PRAGMA foreign_keys;")
foreign_key_violations = query_df("PRAGMA foreign_key_check;")

display(foreign_keys_result)
display(schema_inspection[["name"]])
display(foreign_key_violations)

assert int(foreign_keys_result.iloc[0, 0]) == 1
assert set(schema_inspection["name"]) == EXPECTED_TABLES
assert foreign_key_violations.empty
print("Structure and foreign-key checks passed.")


In [ ]:
# Expected row-count checks.
expected_row_counts = {
    "clinics": 3,
    "patients": 30,
    "encounters": 45,
    "referrals": 40,
    "followups": 40,
}

row_count_records = []
for table, expected_rows in expected_row_counts.items():
    actual_rows = connection.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]
    row_count_records.append({
        "table": table,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "passes": actual_rows == expected_rows,
    })

row_counts_df = pd.DataFrame(row_count_records)
display(row_counts_df)
assert row_counts_df["passes"].all()
print("All expected row-count checks passed.")


In [ ]:
# Content and relationship checks.
content_checks = {
    "encounters_joined_to_patient_and_clinic": """
        SELECT COUNT(*)
        FROM encounters AS e
        JOIN patients AS p ON e.patient_id = p.patient_id
        JOIN clinics AS c ON e.clinic_id = c.clinic_id;
    """,
    "referrals_joined_to_encounter": """
        SELECT COUNT(*)
        FROM referrals AS r
        JOIN encounters AS e ON r.encounter_id = e.encounter_id;
    """,
    "followups_joined_to_referral": """
        SELECT COUNT(*)
        FROM followups AS f
        JOIN referrals AS r ON f.referral_id = r.referral_id;
    """,
}

relationship_results = {
    name: int(query_df(sql).iloc[0, 0])
    for name, sql in content_checks.items()
}
display(pd.Series(relationship_results, name="joined_rows"))

assert relationship_results["encounters_joined_to_patient_and_clinic"] == 45
assert relationship_results["referrals_joined_to_encounter"] == 40
assert relationship_results["followups_joined_to_referral"] == 40

invalid_priorities = query_df("""
    SELECT priority, COUNT(*) AS rows
    FROM referrals
    WHERE priority NOT IN ('Routine', 'High') OR priority IS NULL
    GROUP BY priority;
""")
invalid_social_flags = query_df("""
    SELECT social_needs_flag, COUNT(*) AS rows
    FROM patients
    WHERE social_needs_flag NOT IN (0, 1) OR social_needs_flag IS NULL
    GROUP BY social_needs_flag;
""")
pending_result = query_df("""
    SELECT COUNT(*) AS pending_referrals
    FROM referrals AS r
    JOIN followups AS f ON r.referral_id = f.referral_id
    WHERE f.completion_date IS NULL;
""")

display(invalid_priorities)
display(invalid_social_flags)
display(pending_result)

assert invalid_priorities.empty
assert invalid_social_flags.empty
print("Content and relationship checks passed.")


## Step 7: Recreate the database from a clean file to confirm that the schema and loading process are reproducible.

The function below repeats the schema and loading process in a new database file. It does not copy the original database. Successful row counts and an empty foreign-key check demonstrate that the database can be rebuilt from `schema.sql` and the five source CSVs.


In [ ]:
def build_database(database_path, schema_sql, seed_directory):
    """Create and load a new SQLite database from schema text and CSV files."""
    database_path = Path(database_path)
    seed_directory = Path(seed_directory)

    if database_path.exists():
        database_path.unlink()

    with sqlite3.connect(database_path) as rebuild_connection:
        rebuild_connection.execute("PRAGMA foreign_keys = ON")
        assert rebuild_connection.execute(
            "PRAGMA foreign_keys"
        ).fetchone()[0] == 1

        rebuild_connection.executescript(schema_sql)

        for table in TABLE_LOAD_ORDER:
            csv_path = seed_directory / f"{table}.csv"
            with csv_path.open(newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)
                rows = list(reader)

            columns = list(reader.fieldnames or [])
            placeholders = ", ".join("?" for _ in columns)
            insert_sql = (
                f"INSERT INTO {table} ({', '.join(columns)}) "
                f"VALUES ({placeholders})"
            )
            values = [
                [row[column] if row[column] != "" else None for column in columns]
                for row in rows
            ]
            rebuild_connection.executemany(insert_sql, values)

        rebuild_connection.commit()

        counts = {
            table: rebuild_connection.execute(
                f"SELECT COUNT(*) FROM {table}"
            ).fetchone()[0]
            for table in TABLE_LOAD_ORDER
        }
        violations = rebuild_connection.execute(
            "PRAGMA foreign_key_check"
        ).fetchall()

    return counts, violations


In [ ]:
REBUILD_PATH = WORK_DIR / "community_followup_rebuild.sqlite"

rebuild_counts, rebuild_violations = build_database(
    REBUILD_PATH,
    SCHEMA_SQL,
    SEED_DIR,
)

rebuild_check_df = pd.DataFrame({
    "expected_rows": pd.Series(expected_row_counts),
    "rebuilt_rows": pd.Series(rebuild_counts),
})
rebuild_check_df["matches"] = (
    rebuild_check_df["expected_rows"] == rebuild_check_df["rebuilt_rows"]
)

display(rebuild_check_df)
print("Rebuild foreign-key violations:", rebuild_violations)

assert rebuild_check_df["matches"].all()
assert rebuild_violations == []
print("Clean rebuild passed.")


## Download and submit the required files

The next cell creates `Week12_Submission.zip`. The ZIP contains exactly:

- `community_followup.sqlite`
- `schema.sql`

Before ending the Colab session, download both required submissions:

1. Download the completed notebook with **File → Download → Download .ipynb**.
2. Run the ZIP cell and download `Week12_Submission.zip`.

Submit the completed `Week12_Database_Colab.ipynb` file and `Week12_Submission.zip` in Canvas.


In [ ]:
SUBMISSION_ZIP = WORK_DIR / "Week12_Submission.zip"

connection.commit()

files_to_package = [
    DB_PATH,
    SCHEMA_PATH,
]

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_package:
        assert path.exists(), f"Missing required file: {path.name}"
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(SUBMISSION_ZIP) as archive:
    packaged_names = set(archive.namelist())

assert packaged_names == {"community_followup.sqlite", "schema.sql"}
print("Submission ZIP created:", SUBMISSION_ZIP)
print("ZIP contents:", sorted(packaged_names))

if IN_COLAB:
    files.download(str(SUBMISSION_ZIP))
else:
    print("Download the ZIP from:", SUBMISSION_ZIP)


## Final checklist

- [ ] The completed notebook runs from setup through the clean rebuild.
- [ ] `community_followup.sqlite` contains all five required tables and expected rows.
- [ ] `schema.sql` matches the implemented database design.
- [ ] Foreign-key enforcement returns `1`.
- [ ] `PRAGMA foreign_key_check;` returns no rows.
- [ ] The clean rebuild produces the same row counts.
- [ ] `Week12_Submission.zip` contains only `community_followup.sqlite` and `schema.sql`.
- [ ] The completed `Week12_Database_Colab.ipynb` notebook has been downloaded.
- [ ] Both the completed notebook and `Week12_Submission.zip` are ready to submit in Canvas.
